# Assistants API 概览（Python SDK）

新的 [Assistants API](https://platform.openai.com/docs/assistants/overview) 是我们 [Chat Completions API](https://platform.openai.com/docs/guides/text-generation/chat-completions-api) 的有状态演进版本，旨在简化类助手体验的构建，并让开发者能够使用 Code Interpreter 和 File Search 等强大工具。

![Assistants API 示意图](../images/assistants_overview_diagram.png)

## Chat Completions API 与 Assistants API 对比

**Chat Completions API** 的基本元素是 `Messages`，你可以使用某个 `Model`（`gpt-4o`、`gpt-4o-mini` 等）对其执行 `Completion`。它轻量而强大，但本质上是无状态的，这意味着你必须手动管理对话状态、工具定义、检索文档和代码执行。

**Assistants API** 的基本元素包括：

- `Assistants`，封装了基础模型、指令、工具和（上下文）文档；
- `Threads`，表示对话的状态；以及
- `Runs`，驱动 `Assistant` 在 `Thread` 上的执行，包括文本回复和多步工具使用。

我们将看看如何利用它们来构建强大的有状态体验。


## 设置

### Python SDK

> **注意**
> 我们已更新 [Python SDK](https://github.com/openai/openai-python) 以支持 Assistants API，因此你需要将其更新到最新版本（撰写本文时为 `1.59.4`）。


In [2]:
!pip install --upgrade openai

并运行以下命令确保它是最新版本：


In [3]:
!pip show openai | grep Version

Version: 1.59.4


### 美化输出的辅助函数


In [4]:
import json

def show_json(obj):
    display(json.loads(obj.model_dump_json()))

## 使用 Assistants API 的完整示例


### Assistants


上手 Assistants API 最简单的方式是使用 [Assistants Playground](https://platform.openai.com/playground)。


![Assistants Playground](../images/assistants_overview_assistants_playground.png)


我们先来创建一个助手！我们将像[文档](https://platform.openai.com/docs/assistants/overview)中那样创建一个数学辅导老师（Math Tutor）。


![创建新的 Assistant](../images/assistants_overview_new_assistant.png)


你也可以直接通过 Assistants API 创建 Assistant，如下所示：


In [5]:
from openai import OpenAI
import os

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "<your OpenAI API key if not set as env var>"))


assistant = client.beta.assistants.create(
    name="Math Tutor",
    instructions="You are a personal math tutor. Answer questions briefly, in a sentence or less.",
    model="gpt-4o",
)
show_json(assistant)

{'id': 'asst_qvXmYlZV8zhABI2RtPzDfV6z',
 'created_at': 1736340398,
 'description': None,
 'instructions': 'You are a personal math tutor. Answer questions briefly, in a sentence or less.',
 'metadata': {},
 'model': 'gpt-4o',
 'name': 'Math Tutor',
 'object': 'assistant',
 'tools': [],
 'response_format': 'auto',
 'temperature': 1.0,
 'tool_resources': {'code_interpreter': None, 'file_search': None},
 'top_p': 1.0} 'tools': [],
 'response_format': 'auto',
 'temperature': 1.0,
 'tool_resources': {'code_interpreter': None, 'file_search': None},
 'top_p': 1.0}

无论你是通过 Dashboard 还是通过 API 创建 Assistant，都需要记下 Assistant ID。在 Threads 和 Runs 中，你将通过它来引用你的 Assistant。


接下来，我们将创建一个新的 Thread，并向其中添加一条 Message。它会保存对话的状态，这样我们就不必每次都重新发送完整的消息历史。


### Threads


创建一个新的 thread：


In [6]:
thread = client.beta.threads.create()
show_json(thread)

{'id': 'thread_j4dc1TiHPfkviKUHNi4aAsA6',
 'created_at': 1736340398,
 'metadata': {},
 'object': 'thread',
 'tool_resources': {'code_interpreter': None, 'file_search': None}} 'object': 'thread',
 'tool_resources': {'code_interpreter': None, 'file_search': None}}

然后将 Message 添加到 thread 中：


In [7]:
message = client.beta.threads.messages.create(
    thread_id=thread.id,
    role="user",
    content="I need to solve the equation `3x + 11 = 14`. Can you help me?",
)
show_json(message)

{'id': 'msg_1q4Y7ZZ9gIcPoAKSx9UtrrKJ',
 'assistant_id': None,
 'attachments': [],
 'completed_at': None,
 'attachments': [],
 'completed_at': None,
 'content': [{'text': {'annotations': [],
    'value': 'I need to solve the equation `3x + 11 = 14`. Can you help me?'},
   'type': 'text'}],
 'created_at': 1736340400,
 'incomplete_at': None,
 'incomplete_details': None,
 'metadata': {},
 'object': 'thread.message',
 'role': 'user',
 'run_id': None,
 'status': None,
 'thread_id': 'thread_j4dc1TiHPfkviKUHNi4aAsA6'}

> **注意**
> 尽管你不再需要每次都发送完整的历史记录，但每次 Run 仍会按整个对话历史的 token 数计费。


### Runs

注意，我们创建的 Thread **并没有**与之前创建的 Assistant 关联！Threads 独立于 Assistants 存在，如果你用过 ChatGPT（其中 thread 与某个模型/GPT 绑定），这可能与你的预期不同。

要从 Assistant 获取针对某个 Thread 的回复，我们必须创建一个 Run。创建 Run 会告诉 Assistant 去查看 Thread 中的消息并采取行动：要么添加一条回复，要么使用工具。

> **注意**
> Runs 是 Assistants API 与 Chat Completions API 之间的一个关键区别。在 Chat Completions 中，模型只会回复一条消息；而在 Assistants API 中，一次 Run 可能会让 Assistant 使用一个或多个工具，并可能向 Thread 中添加多条消息。

为了让 Assistant 回复用户，我们来创建 Run。如前所述，你必须_同时_指定 Assistant 和 Thread。


In [8]:
run = client.beta.threads.runs.create(
    thread_id=thread.id,
    assistant_id=assistant.id,
)
show_json(run)

{'id': 'run_qVYsWok6OCjHxkajpIrdHuVP',
 'assistant_id': 'asst_qvXmYlZV8zhABI2RtPzDfV6z',
 'cancelled_at': None,
 'completed_at': None,
 'created_at': 1736340403,
 'expires_at': 1736341003,
 'failed_at': None,
 'incomplete_details': None,
 'incomplete_details': None,
 'instructions': 'You are a personal math tutor. Answer questions briefly, in a sentence or less.',
 'last_error': None,
 'max_completion_tokens': None,
 'max_prompt_tokens': None,
 'max_completion_tokens': None,
 'max_prompt_tokens': None,
 'metadata': {},
 'model': 'gpt-4o',
 'object': 'thread.run',
 'parallel_tool_calls': True,
 'parallel_tool_calls': True,
 'required_action': None,
 'response_format': 'auto',
 'response_format': 'auto',
 'started_at': None,
 'status': 'queued',
 'thread_id': 'thread_j4dc1TiHPfkviKUHNi4aAsA6',
 'tool_choice': 'auto',
 'tools': [],
 'truncation_strategy': {'type': 'auto', 'last_messages': None},
 'usage': None,
 'temperature': 1.0,
 'top_p': 1.0,
 'tool_resources': {}}

与在 Chat Completions API 中创建补全不同，**创建 Run 是一个异步操作**。它会立即返回 Run 的元数据，其中包括一个 `status`，其初始值为 `queued`。随着 Assistant 执行操作（如使用工具和添加消息），`status` 会随之更新。

为了知道 Assistant 何时处理完毕，我们可以在循环中轮询 Run。（流式支持即将推出！）虽然这里我们只检查 `queued` 或 `in_progress` 状态，但在实践中，Run 可能会经历[多种状态变化](https://platform.openai.com/docs/api-reference/runs/object#runs/object-status)，你可以选择将其展示给用户。（这些称为 Steps，稍后会介绍。）


In [9]:
import time

def wait_on_run(run, thread):
    while run.status == "queued" or run.status == "in_progress":
        run = client.beta.threads.runs.retrieve(
            thread_id=thread.id,
            run_id=run.id,
        )
        time.sleep(0.5)
    return run

In [10]:
run = wait_on_run(run, thread)
show_json(run)

{'id': 'run_qVYsWok6OCjHxkajpIrdHuVP',
 'assistant_id': 'asst_qvXmYlZV8zhABI2RtPzDfV6z',
 'cancelled_at': None,
 'completed_at': 1736340406,
 'created_at': 1736340403,
 'expires_at': None,
 'failed_at': None,
 'incomplete_details': None,
 'incomplete_details': None,
 'instructions': 'You are a personal math tutor. Answer questions briefly, in a sentence or less.',
 'last_error': None,
 'max_completion_tokens': None,
 'max_prompt_tokens': None,
 'max_completion_tokens': None,
 'max_prompt_tokens': None,
 'metadata': {},
 'model': 'gpt-4o',
 'object': 'thread.run',
 'parallel_tool_calls': True,
 'parallel_tool_calls': True,
 'required_action': None,
 'response_format': 'auto',
 'started_at': 1736340405,
 'status': 'completed',
 'thread_id': 'thread_j4dc1TiHPfkviKUHNi4aAsA6',
 'tool_choice': 'auto',
 'tools': [],
 'truncation_strategy': {'type': 'auto', 'last_messages': None},
 'usage': {'completion_tokens': 35,
  'prompt_tokens': 66,
  'total_tokens': 101,
  'prompt_token_details': {'cac

### Messages


Run 完成后，我们可以列出 Thread 中的 Messages，看看 Assistant 添加了什么内容。


In [11]:
messages = client.beta.threads.messages.list(thread_id=thread.id)
show_json(messages)

{'data': [{'id': 'msg_A5eAN6ZAJDmFBOYutEm5DFCy',
   'assistant_id': 'asst_qvXmYlZV8zhABI2RtPzDfV6z',
   'attachments': [],
   'completed_at': None,
   'content': [{'text': {'annotations': [],
      'value': 'Sure! Subtract 11 from both sides to get \\(3x = 3\\), then divide by 3 to find \\(x = 1\\).'},
     'type': 'text'}],
   'created_at': 1736340405,
   'incomplete_at': None,
   'incomplete_details': None,
   'metadata': {},
   'object': 'thread.message',
   'role': 'assistant',
   'run_id': 'run_qVYsWok6OCjHxkajpIrdHuVP',
   'status': None,
   'thread_id': 'thread_j4dc1TiHPfkviKUHNi4aAsA6'},
  {'id': 'msg_1q4Y7ZZ9gIcPoAKSx9UtrrKJ',
   'assistant_id': None,
   'attachments': [],
   'completed_at': None,
   'attachments': [],
   'completed_at': None,
   'content': [{'text': {'annotations': [],
      'value': 'I need to solve the equation `3x + 11 = 14`. Can you help me?'},
     'type': 'text'}],
   'created_at': 1736340400,
   'incomplete_at': None,
   'incomplete_details': None,
   

可以看到，Messages 按时间倒序排列——这样做是为了让最新的结果始终位于第一个 `page`（因为结果可能会分页）。请务必留意这一点，因为它与 Chat Completions API 中消息的顺序相反。


我们来让 Assistant 进一步解释一下这个结果！


In [12]:
# Create a message to append to our thread
message = client.beta.threads.messages.create(
    thread_id=thread.id, role="user", content="Could you explain this to me?"
)

# Execute our run
run = client.beta.threads.runs.create(
    thread_id=thread.id,
    assistant_id=assistant.id,
)

# Wait for completion
wait_on_run(run, thread)

# Retrieve all the messages added after our last user message
messages = client.beta.threads.messages.list(
    thread_id=thread.id, order="asc", after=message.id
)
show_json(messages)

{'data': [{'id': 'msg_wSHHvaMnaWktZWsKs6gyoPUB',
   'assistant_id': 'asst_qvXmYlZV8zhABI2RtPzDfV6z',
   'attachments': [],
   'completed_at': None,
   'content': [{'text': {'annotations': [],
      'value': 'Certainly! To isolate \\(x\\), first subtract 11 from both sides of the equation \\(3x + 11 = 14\\), resulting in \\(3x = 3\\). Then, divide both sides by 3 to solve for \\(x\\), giving you \\(x = 1\\).'},
     'type': 'text'}],
   'created_at': 1736340414,
   'incomplete_at': None,
   'incomplete_details': None,
   'metadata': {},
   'object': 'thread.message',
   'role': 'assistant',
   'run_id': 'run_lJsumsDtPTmdG3Enx2CfYrrq',
   'status': None,
   'thread_id': 'thread_j4dc1TiHPfkviKUHNi4aAsA6'}],
 'object': 'list',
 'first_id': 'msg_wSHHvaMnaWktZWsKs6gyoPUB',
 'last_id': 'msg_wSHHvaMnaWktZWsKs6gyoPUB',
 'has_more': False}

为了得到一个回复，这看起来可能步骤很多，尤其是对于这个简单的示例。不过，你很快就会看到，我们几乎不用改动代码，就能为 Assistant 添加非常强大的功能！


### 示例


我们来看看如何把所有这些组合起来。下面是使用你已创建的 Assistant 所需的全部代码。

由于我们已经创建了 Math Assistant，我把它的 ID 保存在 `MATH_ASSISTANT_ID` 中。然后我定义了两个函数：

- `submit_message`：在 Thread 上创建一条 Message，然后启动（并返回）一个新的 Run
- `get_response`：返回 Thread 中的 Messages 列表


In [13]:
from openai import OpenAI

MATH_ASSISTANT_ID = assistant.id  # or a hard-coded ID like "asst-..."

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "<your OpenAI API key if not set as env var>"))

def submit_message(assistant_id, thread, user_message):
    client.beta.threads.messages.create(
        thread_id=thread.id, role="user", content=user_message
    )
    return client.beta.threads.runs.create(
        thread_id=thread.id,
        assistant_id=assistant_id,
    )


def get_response(thread):
    return client.beta.threads.messages.list(thread_id=thread.id, order="asc")

我还定义了一个可复用的 `create_thread_and_run` 函数（它实际上与我们 API 中的组合函数 [`client.beta.threads.create_and_run`](https://platform.openai.com/docs/api-reference/runs/createThreadAndRun) 几乎完全相同 ;) ）。最后，我们可以把每个模拟的用户请求分别提交到一个新的 Thread。

注意，所有这些 API 调用都是异步操作；这意味着我们无需使用异步库（例如 `asyncio`），代码中就实际获得了异步行为！


In [14]:
def create_thread_and_run(user_input):
    thread = client.beta.threads.create()
    run = submit_message(MATH_ASSISTANT_ID, thread, user_input)
    return thread, run


# Emulating concurrent user requests
thread1, run1 = create_thread_and_run(
    "I need to solve the equation `3x + 11 = 14`. Can you help me?"
)
thread2, run2 = create_thread_and_run("Could you explain linear algebra to me?")
thread3, run3 = create_thread_and_run("I don't like math. What can I do?")

# Now all Runs are executing...

所有 Run 都启动后，我们可以逐个等待并获取回复。


In [15]:
import time

# Pretty printing helper
def pretty_print(messages):
    print("# Messages")
    for m in messages:
        print(f"{m.role}: {m.content[0].text.value}")
    print()


# Waiting in a loop
def wait_on_run(run, thread):
    while run.status == "queued" or run.status == "in_progress":
        run = client.beta.threads.runs.retrieve(
            thread_id=thread.id,
            run_id=run.id,
        )
        time.sleep(0.5)
    return run


# Wait for Run 1
run1 = wait_on_run(run1, thread1)
pretty_print(get_response(thread1))

# Wait for Run 2
run2 = wait_on_run(run2, thread2)
pretty_print(get_response(thread2))

# Wait for Run 3
run3 = wait_on_run(run3, thread3)
pretty_print(get_response(thread3))

# Thank our assistant on Thread 3 :)
run4 = submit_message(MATH_ASSISTANT_ID, thread3, "Thank you!")
run4 = wait_on_run(run4, thread3)
pretty_print(get_response(thread3))

# Messages
user: I need to solve the equation `3x + 11 = 14`. Can you help me?
assistant: Sure! Subtract 11 from both sides to get \(3x = 3\), then divide by 3 to find \(x = 1\).

# Messages
user: Could you explain linear algebra to me?
assistant: Linear algebra is the branch of mathematics concerning vector spaces, linear transformations, and systems of linear equations, often represented with matrices.

# Messages
user: I don't like math. What can I do?
assistant: Try relating math to real-life interests or hobbies, practice with fun games or apps, and gradually build confidence with easier problems.

# Messages
user: I don't like math. What can I do?
assistant: Try relating math to real-life interests or hobbies, practice with fun games or apps, and gradually build confidence with easier problems.
user: Thank you!
assistant: You're welcome! If you have any more questions, feel free to ask!



大功告成！

你可能已经注意到，这段代码其实根本不是专门针对数学 Assistant 的……只需更改 Assistant ID，这段代码就适用于你创建的任何新 Assistant！这正是 Assistants API 的强大之处。


## 工具

Assistants API 的一个关键特性是能够为 Assistant 配备工具，如 Code Interpreter、File Search 和自定义 Functions。我们逐一来看。

### Code Interpreter

我们来为 Math Tutor 配备 [Code Interpreter](https://platform.openai.com/docs/assistants/tools/code-interpreter) 工具，可以在 Dashboard 中完成……


![启用 code interpreter](../images/assistants_overview_enable_code_interpreter.png)


……也可以通过 API，使用 Assistant ID 完成。


In [16]:
assistant = client.beta.assistants.update(
    MATH_ASSISTANT_ID,
    tools=[{"type": "code_interpreter"}],
)
show_json(assistant)

{'id': 'asst_qvXmYlZV8zhABI2RtPzDfV6z',
 'created_at': 1736340398,
 'description': None,
 'instructions': 'You are a personal math tutor. Answer questions briefly, in a sentence or less.',
 'metadata': {},
 'model': 'gpt-4o',
 'name': 'Math Tutor',
 'object': 'assistant',
 'tools': [{'type': 'code_interpreter'}],
 'response_format': 'auto',
 'temperature': 1.0,
 'tool_resources': {'code_interpreter': {'file_ids': []}, 'file_search': None},
 'top_p': 1.0} 'tools': [{'type': 'code_interpreter'}],
 'response_format': 'auto',
 'temperature': 1.0,
 'tool_resources': {'code_interpreter': {'file_ids': []}, 'file_search': None},
 'top_p': 1.0}

现在，我们让 Assistant 使用它的新工具。


In [17]:
thread, run = create_thread_and_run(
    "Generate the first 20 fibbonaci numbers with code."
)
run = wait_on_run(run, thread)
pretty_print(get_response(thread))

# Messages
user: Generate the first 20 fibbonaci numbers with code.
assistant: The first 20 Fibonacci numbers are: 0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181.



就是这样！Assistant 在后台使用了 Code Interpreter，并给出了最终回复。

对于某些用例来说，这可能已经足够——不过，如果想更详细地了解 Assistant 究竟做了什么，可以查看 Run 的 Steps。

### Steps


一个 Run 由一个或多个 Step 组成。与 Run 一样，每个 Step 都有一个可查询的 `status`。这对于向用户展示 Step 的进度很有用（例如在 Assistant 编写代码或执行检索时显示加载动画）。


In [18]:
run_steps = client.beta.threads.runs.steps.list(
    thread_id=thread.id, run_id=run.id, order="asc"
)

我们来看看每个 Step 的 `step_details`。


In [19]:
for step in run_steps.data:
    step_details = step.step_details
    print(json.dumps(show_json(step_details), indent=4))

{'tool_calls': [{'id': 'call_E1EE1loDmcWoc7FpkOMKYj6n',
   'code_interpreter': {'input': 'def generate_fibonacci(n):\n    fib_sequence = [0, 1]\n    while len(fib_sequence) < n:\n        next_value = fib_sequence[-1] + fib_sequence[-2]\n        fib_sequence.append(next_value)\n    return fib_sequence\n\n# Generate the first 20 Fibonacci numbers\nfirst_20_fibonacci = generate_fibonacci(20)\nfirst_20_fibonacci',
    'outputs': []},
   'type': 'code_interpreter'}],
 'type': 'tool_calls'}

null


{'message_creation': {'message_id': 'msg_RzTnbBMmzDYHk79a0x9qM5uU'},
 'type': 'message_creation'}

null


我们可以看到两个 Step 的 `step_details`：

1. `tool_calls`（复数，因为单个 Step 中可能有多个）
2. `message_creation`

第一个 Step 是 `tool_calls`，具体使用的是 `code_interpreter`，其中包含：

- `input`，即调用工具前生成的 Python 代码；以及
- `output`，即运行 Code Interpreter 的结果。

第二个 Step 是 `message_creation`，其中包含添加到 Thread 中、用于向用户传达结果的 `message`。


### 文件搜索

Assistants API 中另一个强大的工具是 [File search（文件搜索）](https://platform.openai.com/docs/assistants/tools/file-search)。它允许向 Assistant 上传文件，作为回答问题时使用的知识库。


![启用检索](../images/assistants_overview_enable_retrieval.png)


In [20]:
# Upload the file
file = client.files.create(
    file=open(
        "data/language_models_are_unsupervised_multitask_learners.pdf",
        "rb",
    ),
    purpose="assistants",
)

# Create a vector store
vector_store = client.beta.vector_stores.create(
    name="language_models_are_unsupervised_multitask_learners",
)

# Add the file to the vector store
vector_store_file = client.beta.vector_stores.files.create_and_poll(
    vector_store_id=vector_store.id,
    file_id=file.id,
)

# Confirm the file was added
while vector_store_file.status == "in_progress":
    time.sleep(1)
if vector_store_file.status == "completed":
    print("File added to vector store")
elif vector_store_file.status == "failed":
    raise Exception("Failed to add file to vector store")

# Update Assistant
assistant = client.beta.assistants.update(
    MATH_ASSISTANT_ID,
    tools=[{"type": "code_interpreter"}, {"type": "file_search"}],
    tool_resources={
        "file_search":{
            "vector_store_ids": [vector_store.id]
        },
        "code_interpreter": {
            "file_ids": [file.id]
        }
    },
)
show_json(assistant)

File added to vector store


{'id': 'asst_qvXmYlZV8zhABI2RtPzDfV6z',
 'created_at': 1736340398,
 'description': None,
 'instructions': 'You are a personal math tutor. Answer questions briefly, in a sentence or less.',
 'metadata': {},
 'model': 'gpt-4o',
 'name': 'Math Tutor',
 'object': 'assistant',
 'tools': [{'type': 'code_interpreter'},
  {'type': 'file_search',
   'file_search': {'max_num_results': None,
    'ranking_options': {'score_threshold': 0.0,
     'ranker': 'default_2024_08_21'}}}],
 'response_format': 'auto',
 'temperature': 1.0,
 'tool_resources': {'code_interpreter': {'file_ids': ['file-GQFm2i7N8LrAQatefWKEsE']},
  'file_search': {'vector_store_ids': ['vs_dEArILZSJh7J799QACi3QhuU']}},
 'top_p': 1.0}

In [21]:
thread, run = create_thread_and_run(
    "What are some cool math concepts behind this ML paper pdf? Explain in two sentences."
)
run = wait_on_run(run, thread)
pretty_print(get_response(thread))

# Messages
user: What are some cool math concepts behind this ML paper pdf? Explain in two sentences.
assistant: The paper explores the concept of multitask learning where a single model is used to perform various tasks, modeling the conditional distribution \( p(\text{output} | \text{input, task}) \), inspired by probabilistic approaches【6:10†source】. It also discusses the use of Transformer-based architectures and parallel corpus substitution in language models, enhancing their ability to generalize across domain tasks without explicit task-specific supervision【6:2†source】【6:5†source】.



> **注意**
> File Search 还有更多细节，例如 [Annotations](https://platform.openai.com/docs/assistants/how-it-works/managing-threads-and-messages)，可能会在其他 cookbook 中介绍。


In [22]:
# Delete the vector store
client.beta.vector_stores.delete(vector_store.id)


VectorStoreDeleted(id='vs_dEArILZSJh7J799QACi3QhuU', deleted=True, object='vector_store.deleted')

### Functions

作为 Assistant 的最后一个强大工具，你可以指定自定义 [Functions](https://platform.openai.com/docs/assistants/tools/function-calling)（与 Chat Completions API 中的[函数调用](https://platform.openai.com/docs/guides/function-calling)非常相似）。在 Run 期间，Assistant 可以表明它想调用你指定的一个或多个函数。然后由你负责调用该函数，并将输出返回给 Assistant。

我们通过为 Math Tutor 定义一个 `display_quiz()` 函数来看一个示例。

该函数将接收一个 `title` 和一个 `question` 数组，展示测验，并为以下每一项获取用户输入：

- `title`
- `questions`
  - `question_text`
  - `question_type`: [`MULTIPLE_CHOICE`, `FREE_RESPONSE`]
  - `choices`: ["choice 1", "choice 2", ...]

我会用 `get_mock_response...` 来模拟回复。在实际应用中，这里将获取用户的真实输入。


In [24]:
def get_mock_response_from_user_multiple_choice():
    return "a"


def get_mock_response_from_user_free_response():
    return "I don't know."


def display_quiz(title, questions):
    print("Quiz:", title)
    print()
    responses = []

    for q in questions:
        print(q["question_text"])
        response = ""

        # If multiple choice, print options
        if q["question_type"] == "MULTIPLE_CHOICE":
            for i, choice in enumerate(q["choices"]):
                print(f"{i}. {choice}")
            response = get_mock_response_from_user_multiple_choice()

        # Otherwise, just get response
        elif q["question_type"] == "FREE_RESPONSE":
            response = get_mock_response_from_user_free_response()

        responses.append(response)
        print()

    return responses

下面是一个示例测验的样子：


In [25]:
responses = display_quiz(
    "Sample Quiz",
    [
        {"question_text": "What is your name?", "question_type": "FREE_RESPONSE"},
        {
            "question_text": "What is your favorite color?",
            "question_type": "MULTIPLE_CHOICE",
            "choices": ["Red", "Blue", "Green", "Yellow"],
        },
    ],
)
print("Responses:", responses)

Quiz: Sample Quiz

What is your name?

What is your favorite color?
0. Red
1. Blue
2. Green
3. Yellow

Responses: ["I don't know.", 'a']


现在，我们以 JSON 格式定义该函数的接口，以便 Assistant 调用它：


In [37]:
function_json = {
    "name": "display_quiz",
    "description": "Displays a quiz to the student, and returns the student's response. A single quiz can have multiple questions.",
    "parameters": {
        "type": "object",
        "properties": {
            "title": {"type": "string"},
            "questions": {
                "type": "array",
                "description": "An array of questions, each with a title and potentially options (if multiple choice).",
                "items": {
                    "type": "object",
                    "properties": {
                        "question_text": {"type": "string"},
                        "question_type": {
                            "type": "string",
                            "enum": ["MULTIPLE_CHOICE", "FREE_RESPONSE"]
                        },
                        "choices": {"type": "array", "items": {"type": "string"}}
                    },
                    "required": ["question_text"]
                }
            }
        },
        "required": ["title", "questions"]
    }
}


同样，我们通过 Dashboard 或 API 更新 Assistant。


![启用自定义函数](../images/assistants_overview_enable_function.png)

> **注意**
> 由于缩进等原因，将函数 JSON 粘贴到 Dashboard 中有点麻烦。我直接让 ChatGPT 按照 Dashboard 上某个示例的格式来格式化我的函数 :)。


In [38]:
assistant = client.beta.assistants.update(
    MATH_ASSISTANT_ID,
    tools=[
        {"type": "code_interpreter"},
        {"type": "file_search"},
        {"type": "function", "function": function_json},
    ],
)
show_json(assistant)

{'id': 'asst_qvXmYlZV8zhABI2RtPzDfV6z',
 'created_at': 1736340398,
 'description': None,
 'instructions': 'You are a personal math tutor. Answer questions briefly, in a sentence or less.',
 'metadata': {},
 'model': 'gpt-4o',
 'name': 'Math Tutor',
 'object': 'assistant',
 'tools': [{'type': 'code_interpreter'},
  {'type': 'file_search',
   'file_search': {'max_num_results': None,
    'ranking_options': {'score_threshold': 0.0,
     'ranker': 'default_2024_08_21'}}},
  {'function': {'name': 'display_quiz',
    'description': "Displays a quiz to the student, and returns the student's response. A single quiz can have multiple questions.",
    'description': "Displays a quiz to the student, and returns the student's response. A single quiz can have multiple questions.",
    'parameters': {'type': 'object',
     'properties': {'title': {'type': 'string'},
      'questions': {'type': 'array',
       'description': 'An array of questions, each with a title and potentially options (if multipl

现在，我们来请求一个测验。


In [39]:
thread, run = create_thread_and_run(
    "Make a quiz with 2 questions: One open ended, one multiple choice. Then, give me feedback for the responses."
)
run = wait_on_run(run, thread)
run.status

'requires_action'

然而，现在检查 Run 的 `status` 时，我们看到的是 `requires_action`！我们来仔细看看。


In [40]:
show_json(run)

{'id': 'run_ekMRSI2h35asEzKirRf4BTwZ',
 'assistant_id': 'asst_qvXmYlZV8zhABI2RtPzDfV6z',
 'cancelled_at': None,
 'completed_at': None,
 'created_at': 1736341020,
 'expires_at': 1736341620,
 'failed_at': None,
 'incomplete_details': None,
 'incomplete_details': None,
 'instructions': 'You are a personal math tutor. Answer questions briefly, in a sentence or less.',
 'last_error': None,
 'max_completion_tokens': None,
 'max_prompt_tokens': None,
 'max_completion_tokens': None,
 'max_prompt_tokens': None,
 'metadata': {},
 'model': 'gpt-4o',
 'object': 'thread.run',
 'parallel_tool_calls': True,
 'required_action': {'submit_tool_outputs': {'tool_calls': [{'id': 'call_uvJEn0fxM4sgmzek8wahBGLi',
     'function': {'arguments': '{"title":"Math Quiz","questions":[{"question_text":"What is the derivative of the function f(x) = 3x^2 + 2x - 5?","question_type":"FREE_RESPONSE"},{"question_text":"What is the value of \\\\( \\\\int_{0}^{1} 2x \\\\, dx \\\\)?","question_type":"MULTIPLE_CHOICE","choic

`required_action` 字段表明有一个工具正在等待我们运行它，并将其输出提交回 Assistant。具体来说，就是 `display_quiz` 函数！我们先来解析 `name` 和 `arguments`。

> **注意**
> 虽然在这个例子中我们知道只有一次工具调用，但在实践中 Assistant 可能会选择调用多个工具。


In [41]:
# Extract single tool call
tool_call = run.required_action.submit_tool_outputs.tool_calls[0]
name = tool_call.function.name
arguments = json.loads(tool_call.function.arguments)

print("Function Name:", name)
print("Function Arguments:")
arguments

Function Name: display_quiz
Function Arguments:


{'title': 'Math Quiz',
 'questions': [{'question_text': 'What is the derivative of the function f(x) = 3x^2 + 2x - 5?',
   'question_type': 'FREE_RESPONSE'},
  {'question_text': 'What is the value of \\( \\int_{0}^{1} 2x \\, dx \\)?',
   'question_type': 'MULTIPLE_CHOICE',
   'choices': ['0', '1', '2', '3']}]}

现在，我们用 Assistant 提供的参数实际调用 `display_quiz` 函数：


In [42]:
responses = display_quiz(arguments["title"], arguments["questions"])
print("Responses:", responses)

Quiz: Math Quiz
Quiz: Math Quiz

What is the derivative of the function f(x) = 3x^2 + 2x - 5?

What is the value of \( \int_{0}^{1} 2x \, dx \)?
0. 0
1. 1
2. 2
3. 3

Responses: ["I don't know.", 'a']


太好了！（请记住，这些回复是我们之前模拟的。在实际情况中，我们会从这次函数调用中获取用户的真实输入。）

拿到回复后，我们将它们提交回 Assistant。我们需要 `tool_call` ID，它可以在之前解析出的 `tool_call` 中找到。我们还需要把回复的 `list` 编码为 `str`。


In [43]:
run = client.beta.threads.runs.submit_tool_outputs(
    thread_id=thread.id,
    run_id=run.id,
    tool_outputs=tool_outputs
)
show_json(run)

{'id': 'run_ekMRSI2h35asEzKirRf4BTwZ',
 'assistant_id': 'asst_qvXmYlZV8zhABI2RtPzDfV6z',
 'cancelled_at': None,
 'completed_at': None,
 'created_at': 1736341020,
 'expires_at': 1736341620,
 'failed_at': None,
 'incomplete_details': None,
 'incomplete_details': None,
 'instructions': 'You are a personal math tutor. Answer questions briefly, in a sentence or less.',
 'last_error': None,
 'max_completion_tokens': None,
 'max_prompt_tokens': None,
 'max_completion_tokens': None,
 'max_prompt_tokens': None,
 'metadata': {},
 'model': 'gpt-4o',
 'object': 'thread.run',
 'parallel_tool_calls': True,
 'parallel_tool_calls': True,
 'required_action': None,
 'response_format': 'auto',
 'started_at': 1736341022,
 'status': 'queued',
 'thread_id': 'thread_8bK2PXfoeijEHBVEzYuJXt17',
 'tool_choice': 'auto',
 'tools': [{'type': 'code_interpreter'},
  {'type': 'file_search',
   'file_search': {'max_num_results': None,
    'ranking_options': {'score_threshold': 0.0,
     'ranker': 'default_2024_08_21'}

现在可以再次等待 Run 完成，然后查看我们的 Thread！


In [44]:
run = wait_on_run(run, thread)
pretty_print(get_response(thread))

# Messages
user: Make a quiz with 2 questions: One open ended, one multiple choice. Then, give me feedback for the responses.
assistant: Since no specific information was found in the uploaded file, I'll create a general math quiz for you:

1. **Open-ended Question**: What is the derivative of the function \( f(x) = 3x^2 + 2x - 5 \)?

2. **Multiple Choice Question**: What is the value of \( \int_{0}^{1} 2x \, dx \)?
    - A) 0
    - B) 1
    - C) 2
    - D) 3

I will now present the quiz to you for response.
assistant: Here is the feedback for your responses:

1. **Derivative Question**: 
   - Your Response: "I don't know."
   - Feedback: The derivative of \( f(x) = 3x^2 + 2x - 5 \) is \( f'(x) = 6x + 2 \).

2. **Integration Question**: 
   - Your Response: A) 0
   - Feedback: The correct answer is B) 1. The integration \(\int_{0}^{1} 2x \, dx \) evaluates to 1.



耶 🎉


## 结论

我们在本 notebook 中涵盖了大量内容，给自己击个掌吧！希望你现在已经打下了坚实的基础，可以借助 Code Interpreter、Retrieval 和 Functions 等工具构建强大的有状态体验！

为了简洁起见，有几个部分我们没有涉及，下面是一些可供进一步探索的资源：

- [Annotations](https://platform.openai.com/docs/assistants/how-it-works/managing-threads-and-messages)：解析文件引用
- [Files](https://platform.openai.com/docs/api-reference/assistants/file-object)：Thread 作用域与 Assistant 作用域
- [并行函数调用](https://platform.openai.com/docs/guides/function-calling/parallel-function-calling)：在单个 Step 中调用多个工具
- 多 Assistant 的 Thread Runs：单个 Thread 中包含来自多个 Assistant 的消息
- 流式输出：即将推出！

现在就去构建一些令人惊[叹](https://www.youtube.com/watch?v=xvFZjo5PgG0&pp=ygUQcmljayByb2xsIG5vIGFkcw%3D%3D)的东西吧！